# ISIC 2019 — прогон бейслайна на Kaggle

Ноутбук встраивает код проекта (`src/`, `configs/`, `scripts/`) напрямую,
ничего качать/клонировать не надо.

## Что нужно сделать один раз в Kaggle UI

1. **Accelerator** → GPU T4 x1 (или P100).
2. **Add Input** → датасет `andrewmvd/isic-2019` (или любой с файлами
   `ISIC_2019_Training_GroundTruth.csv`, `ISIC_2019_Training_Metadata.csv` и
   папкой изображений).
3. **Run All**.

После завершения — скачать `/kaggle/working/isic_outputs.zip` и распаковать
локально в `reports/` + `runs/`.


In [ ]:
import sys, torch
print("python:", sys.version.split()[0])
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))


In [ ]:
import os
for sub in ("src", "scripts", "configs", "data", "runs", "reports"):
    os.makedirs("%s/" % "/kaggle/working/isic_project" + sub, exist_ok=True)
%cd /kaggle/working/isic_project


In [ ]:
%%writefile /kaggle/working/isic_project/configs/baseline.yaml
# Базовая конфигурация эксперимента ISIC-2019.
# Один файл управляет всем пайплайном: train -> evaluate -> stress -> аналитика.
# Пути разрешаются относительно корня проекта (см. src/config.py::load_config).
# Любой путь можно переопределить переменными окружения ISIC_DATA_DIR / ISIC_RUNS_DIR /
# ISIC_REPORTS_DIR (нужно для запуска на Kaggle, где рабочая папка другая).

seed: 42
name:                       # суффикс имени прогона; пусто -> только метка времени

paths:
  data_dir: data             # база; images_dir/ground_truth ниже — относительно неё
  images_dir: images
  ground_truth: ground_truth.csv
  runs_dir: runs
  reports_dir: reports
  checkpoint: baseline_resnet18.pth   # относительно runs_dir

# Порядок классов фиксирован: индекс класса = позиция в списке.
classes: [MEL, NV, BCC, AK, BKL, DF, VASC, SCC]

split:
  group_by: lesion_id        # группировка против утечки кадров одного поражения
  val_size: 0.2
  random_state: 42

model:
  backbone: resnet18
  pretrained: true
  image_size: 224

train:
  epochs: 5
  batch_size: 32
  lr: 1.0e-4
  optimizer: adam
  weight_decay: 0.0          # намеренно 0: бейслайн без регуляризации
  num_workers: 4

eval:
  batch_size: 64
  num_workers: 4

# Решающее правило для критичного узла «подозрение на меланому».
decision:
  positive_class: MEL
  malignant_classes: [MEL, BCC, SCC]
  threshold_step: 0.05
  cost_fp: 1                 # цена ложной тревоги
  cost_fn: 10               # цена пропуска меланомы (на порядок дороже)
  default_threshold: 0.5     # технический дефолт (эквивалент arg-max для бинарного MEL)
  # Веса цены пропуска по классам для per-class риска (sensitivity-проверка).
  per_class_fn_costs: {MEL: 10, BCC: 8, SCC: 8, AK: 3, NV: 1, BKL: 1, DF: 1, VASC: 1}

stress:
  subset:                    # пусто -> вся валидация; число -> детерминированный сабсэмпл
  seed: 42

calibration:
  n_bins: 10


In [ ]:
%%writefile /kaggle/working/isic_project/src/__init__.py
"""ISIC baseline package."""


In [ ]:
%%writefile /kaggle/working/isic_project/src/utils.py
"""Мелкие хелперы пайплайна: пути, имена прогонов, сериализация.

Держим всё в одном месте, чтобы остальные модули не тащили дублирующийся
boilerplate. Ничего тяжёлого тут не импортируется.
"""
import json
from datetime import datetime
from pathlib import Path

import yaml

# Корень проекта = родитель папки src/.
PROJECT_ROOT = Path(__file__).resolve().parent.parent


def as_path(value, base=PROJECT_ROOT):
    """Привести значение к абсолютному Path относительно base (если оно не абсолютное)."""
    path = Path(value)
    return path if path.is_absolute() else (Path(base) / path).resolve()


def run_stamp(suffix=None):
    """Имя прогона: метка времени + опциональный суффикс."""
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    suffix = (suffix or "").strip().replace(" ", "_")
    return f"{stamp}_{suffix}" if suffix else stamp


def ensure_dir(path):
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def dump_json(path, payload):
    path = Path(path)
    ensure_dir(path.parent)
    path.write_text(json.dumps(jsonable(payload), ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    return path


def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def dump_yaml(path, payload):
    path = Path(path)
    ensure_dir(path.parent)
    path.write_text(yaml.safe_dump(jsonable(payload), allow_unicode=True, sort_keys=False), encoding="utf-8")
    return path


def first_set(*values, default=None):
    """Вернуть первое не-None значение (CLI -> конфиг -> дефолт)."""
    for value in values:
        if value is not None:
            return value
    return default


def jsonable(value):
    """Рекурсивно превратить numpy/Path/множества в сериализуемые типы."""
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(k): jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [jsonable(v) for v in value]
    if hasattr(value, "tolist"):           # numpy array / тензор
        return value.tolist()
    if hasattr(value, "item"):             # numpy скаляр
        try:
            return value.item()
        except (ValueError, TypeError):
            pass
    return str(value)


In [ ]:
%%writefile /kaggle/working/isic_project/src/config.py
"""Загрузка и нормализация конфигурации эксперимента.

Один YAML-файл (`configs/baseline.yaml`) — единственный источник правды для
путей, гиперпараметров обучения, оценки и решающего правила. Здесь он читается,
дополняется дефолтами и в нём резолвятся пути. Переменные окружения
ISIC_DATA_DIR / ISIC_RUNS_DIR / ISIC_REPORTS_DIR имеют приоритет над файлом —
это нужно для Kaggle, где рабочая директория не совпадает с корнем репозитория.
"""
import copy
import os
from pathlib import Path

import yaml

from src.utils import PROJECT_ROOT, as_path

DEFAULT_CONFIG = PROJECT_ROOT / "configs" / "baseline.yaml"

_DEFAULTS = {
    "seed": 42,
    "name": None,
    "classes": ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"],
    "split": {"group_by": "lesion_id", "val_size": 0.2, "random_state": 42},
    "model": {"backbone": "resnet18", "pretrained": True, "image_size": 224},
    "train": {"epochs": 5, "batch_size": 32, "lr": 1e-4, "optimizer": "adam",
              "weight_decay": 0.0, "num_workers": 4},
    "eval": {"batch_size": 64, "num_workers": 4},
    "decision": {"positive_class": "MEL", "malignant_classes": ["MEL", "BCC", "SCC"],
                 "threshold_step": 0.05, "cost_fp": 1, "cost_fn": 10, "default_threshold": 0.5,
                 "per_class_fn_costs": {}},
    "stress": {"subset": None, "seed": 42},
    "calibration": {"n_bins": 10},
}


def _merge(base, override):
    """Глубокое слияние словарей: значения из override побеждают."""
    out = copy.deepcopy(base)
    for key, value in (override or {}).items():
        if isinstance(value, dict) and isinstance(out.get(key), dict):
            out[key] = _merge(out[key], value)
        else:
            out[key] = value
    return out


def load_config(config_path=None):
    config_path = Path(config_path) if config_path else DEFAULT_CONFIG
    raw = yaml.safe_load(config_path.read_text(encoding="utf-8")) or {}
    cfg = _merge(_DEFAULTS, raw)
    cfg["config_path"] = str(config_path.resolve())

    # Базовые каталоги: env-var > файл > дефолт.
    paths = cfg.setdefault("paths", {})
    data_dir = os.environ.get("ISIC_DATA_DIR", paths.get("data_dir", "data"))
    runs_dir = os.environ.get("ISIC_RUNS_DIR", paths.get("runs_dir", "runs"))
    reports_dir = os.environ.get("ISIC_REPORTS_DIR", paths.get("reports_dir", "reports"))

    paths["data_dir"] = str(as_path(data_dir))
    paths["runs_dir"] = str(as_path(runs_dir))
    paths["reports_dir"] = str(as_path(reports_dir))
    paths["images_dir"] = str(as_path(paths.get("images_dir", "images"), paths["data_dir"]))
    paths["ground_truth"] = str(as_path(paths.get("ground_truth", "ground_truth.csv"), paths["data_dir"]))
    paths["checkpoint"] = str(as_path(paths.get("checkpoint", "baseline_resnet18.pth"), paths["runs_dir"]))

    # Индекс позитивного (критичного) класса для бинаризации.
    classes = cfg["classes"]
    decision = cfg["decision"]
    decision["positive_index"] = classes.index(decision["positive_class"])
    decision["malignant_indices"] = [classes.index(c) for c in decision.get("malignant_classes", [])]

    return cfg


def num_classes(cfg):
    return len(cfg["classes"])


In [ ]:
%%writefile /kaggle/working/isic_project/src/dataset.py
"""Датасет ISIC и групповой сплит без утечки между train/val.

Главная задача модуля — гарантировать, что разные кадры одного поражения
(`lesion_id`) не попадают одновременно в train и val. На ISIC это типичная
ситуация (несколько ракурсов одного образца), и случайный построчный сплит
завышал бы валидационные метрики.
"""
import os

import pandas as pd
import torch
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import Dataset
from torchvision import transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

_SUFFIXES = (".jpg", ".jpeg", ".png")


class ISICDataset(Dataset):
    """Чтение (image, label) из таблицы ground_truth.csv."""

    def __init__(self, frame, images_dir, transform=None):
        self.frame = frame.reset_index(drop=True)
        self.images_dir = images_dir
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def _resolve_image(self, name):
        for suffix in _SUFFIXES:
            candidate = os.path.join(self.images_dir, name + suffix)
            if os.path.isfile(candidate):
                return candidate
        # последний шанс — имя уже с расширением
        direct = os.path.join(self.images_dir, name)
        if os.path.isfile(direct):
            return direct
        raise FileNotFoundError(f"не найдено изображение для '{name}' в {self.images_dir}")

    def image_path(self, idx):
        return self._resolve_image(str(self.frame.iloc[idx]["image"]))

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image = Image.open(self._resolve_image(str(row["image"]))).convert("RGB")
        label = torch.tensor(int(row["label"]), dtype=torch.long)
        if self.transform is not None:
            image = self.transform(image)
        return image, label


def eval_transform(image_size=224):
    """Детерминированный препроцессинг для оценки/инференса (без аугментаций)."""
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


# train в бейслайне намеренно совпадает с eval — никаких train-time аугментаций.
train_transform = eval_transform


def load_table(csv_path):
    return pd.read_csv(csv_path)


def split_frame(frame, group_by="lesion_id", val_size=0.2, random_state=42):
    """Групповой train/val сплит. Если колонки группировки нет — fallback на image."""
    if group_by in frame.columns:
        groups = frame[group_by].fillna(frame["image"]).astype(str)
    else:
        groups = frame["image"].astype(str)
    splitter = GroupShuffleSplit(n_splits=1, test_size=val_size, random_state=random_state)
    train_idx, val_idx = next(splitter.split(frame, groups=groups))
    return frame.iloc[train_idx].copy(), frame.iloc[val_idx].copy()


def get_splits(cfg):
    """Прочитать таблицу и вернуть (train_df, val_df) по настройкам конфига."""
    frame = load_table(cfg["paths"]["ground_truth"])
    split = cfg["split"]
    return split_frame(frame, split["group_by"], split["val_size"], split["random_state"])


def pick_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


In [ ]:
%%writefile /kaggle/working/isic_project/src/model.py
"""Сборка классификатора поражений кожи.

Бэкбон выбирается по имени из конфига. По умолчанию — ResNet18, предобученный
на ImageNet, с заменой последнего слоя на нужное число классов. Простой выбор
сознателен: цель бейслайна — выявить узкие места задачи, а не выжать SOTA.
"""
import torch
from torchvision import models

# Имя бэкбона -> (конструктор, веса по умолчанию, имя атрибута классификатора).
_BACKBONES = {
    "resnet18": (models.resnet18, getattr(models, "ResNet18_Weights", None), "fc"),
    "resnet34": (models.resnet34, getattr(models, "ResNet34_Weights", None), "fc"),
    "resnet50": (models.resnet50, getattr(models, "ResNet50_Weights", None), "fc"),
}


def build_model(backbone="resnet18", num_classes=8, pretrained=True):
    if backbone not in _BACKBONES:
        raise ValueError(f"неизвестный бэкбон: {backbone!r}; доступны {list(_BACKBONES)}")
    ctor, weights_enum, head_attr = _BACKBONES[backbone]

    weights = weights_enum.DEFAULT if (pretrained and weights_enum is not None) else None
    model = ctor(weights=weights)

    head = getattr(model, head_attr)
    setattr(model, head_attr, torch.nn.Linear(head.in_features, num_classes))
    return model


def load_checkpoint(model, checkpoint_path, device):
    state = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state)
    return model


In [ ]:
%%writefile /kaggle/working/isic_project/src/metrics.py
"""Метрические примитивы для решающего правила и калибровки.

Здесь — чистые функции над numpy-массивами вероятностей и меток, без модели и
ввода-вывода. Их переиспользуют threshold-sweep, выбор operating point,
калибровка и анализ ошибок.
"""
import numpy as np


def binary_counts(scores, positive, tau):
    """TP/FP/FN/TN для бинаризации «класс vs всё» по порогу tau.

    scores   — вероятность позитивного класса, shape (N,)
    positive — бинарная истина (1 = позитив), shape (N,)
    """
    pred = scores >= tau
    pos = positive.astype(bool)
    tp = int(np.sum(pred & pos))
    fp = int(np.sum(pred & ~pos))
    fn = int(np.sum(~pred & pos))
    tn = int(np.sum(~pred & ~pos))
    return {"tp": tp, "fp": fp, "fn": fn, "tn": tn}


def rates(counts):
    tp, fp, fn, tn = counts["tp"], counts["fp"], counts["fn"], counts["tn"]
    recall = tp / max(1, tp + fn)
    precision = tp / max(1, tp + fp)
    specificity = tn / max(1, tn + fp)
    fpr = fp / max(1, fp + tn)
    f1 = 2 * precision * recall / max(1e-9, precision + recall)
    return {"recall": recall, "precision": precision, "specificity": specificity,
            "fpr": fpr, "f1": f1}


def uniform_risk(counts, cost_fp, cost_fn):
    """Скалярная цена ошибок при одинаковой стоимости внутри типа."""
    return cost_fp * counts["fp"] + cost_fn * counts["fn"]


def threshold_grid(step=0.05):
    n = int(round(1.0 / step)) + 1
    return [round(i * step, 4) for i in range(n)]


def sweep(scores, positive, taus, cost_fp=1, cost_fn=10):
    """Прогон по сетке порогов: счётчики + ставки + риск на каждом tau."""
    rows = []
    for tau in taus:
        counts = binary_counts(scores, positive, tau)
        row = {"threshold": tau, **counts, **rates(counts),
               "risk": uniform_risk(counts, cost_fp, cost_fn)}
        rows.append(row)
    return rows


def reliability_bins(confidences, correct, n_bins=10):
    """Бины для reliability diagram + ECE.

    confidences — уверенность предсказания (max softmax или скор позитива)
    correct     — 1, если предсказание верно
    """
    confidences = np.asarray(confidences, dtype=float)
    correct = np.asarray(correct, dtype=float)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    total = max(1, len(confidences))
    ece = 0.0
    bins = []
    for i in range(n_bins):
        low, high = edges[i], edges[i + 1]
        mask = (confidences >= low) & (confidences < high if i < n_bins - 1 else confidences <= high)
        count = int(np.sum(mask))
        if count == 0:
            bins.append({"low": float(low), "high": float(high), "count": 0,
                         "acc": None, "conf": None})
            continue
        acc = float(np.mean(correct[mask]))
        conf = float(np.mean(confidences[mask]))
        ece += (count / total) * abs(acc - conf)
        bins.append({"low": float(low), "high": float(high), "count": count,
                     "acc": acc, "conf": conf})
    return {"n_bins": n_bins, "ece": float(ece), "bins": bins}


def macro_f1_from_confusion(confusion):
    """Macro-F1 из матрицы ошибок (строки — истина, столбцы — предсказание)."""
    confusion = np.asarray(confusion, dtype=float)
    f1s = []
    for c in range(confusion.shape[0]):
        tp = confusion[c, c]
        fp = confusion[:, c].sum() - tp
        fn = confusion[c, :].sum() - tp
        precision = tp / max(1e-9, tp + fp)
        recall = tp / max(1e-9, tp + fn)
        f1s.append(2 * precision * recall / max(1e-9, precision + recall))
    return float(np.mean(f1s))


In [ ]:
%%writefile /kaggle/working/isic_project/src/transforms.py
"""Параметризованные искажения входного кадра для стресс-теста.

Каждое преобразование T_alpha — детерминированная функция одного параметра,
действующая на одно PIL-изображение (до нормализации). Сетки параметров и
«коридоры инвариантности» вынесены в общие структуры, чтобы стресс-тест и его
анализ опирались на один источник правды.

Семейства подобраны под реальные искажения дерматоскопии: освещение и баланс
камеры (brightness/contrast/gamma), цветовой сдвиг устройства (hue), потеря
резкости (blur), шум сенсора (noise), сжатие (jpeg) и частичное перекрытие
поражения гелем/волосом/линейкой (occlusion).
"""
import io
import random

import numpy as np
from PIL import Image, ImageEnhance, ImageFilter


def brightness(image, alpha):
    # alpha в стопах экспозиции (EV); 0 — без изменений.
    if alpha == 0:
        return image
    return ImageEnhance.Brightness(image).enhance(2 ** alpha)


def contrast(image, alpha):
    if alpha == 1.0:
        return image
    return ImageEnhance.Contrast(image).enhance(alpha)


def gamma(image, alpha):
    if alpha == 1.0:
        return image
    arr = np.asarray(image).astype(np.float32) / 255.0
    arr = np.clip(arr ** alpha, 0.0, 1.0)
    return Image.fromarray((arr * 255).astype(np.uint8))


def hue_shift(image, alpha):
    # alpha — сдвиг тона в градусах (0..255 в пространстве PIL H-канала).
    if alpha == 0:
        return image
    hsv = np.asarray(image.convert("HSV")).astype(np.int16)
    hsv[..., 0] = (hsv[..., 0] + int(alpha)) % 256
    return Image.fromarray(hsv.astype(np.uint8), mode="HSV").convert("RGB")


def gaussian_noise(image, alpha):
    # alpha — sigma шума в шкале uint8.
    if alpha == 0:
        return image
    rng = np.random.default_rng(42)
    arr = np.asarray(image).astype(np.float32)
    noisy = arr + rng.normal(0.0, alpha, arr.shape)
    return Image.fromarray(np.clip(noisy, 0, 255).astype(np.uint8))


def gaussian_blur(image, alpha):
    # alpha — радиус размытия в пикселях.
    if alpha == 0:
        return image
    return image.filter(ImageFilter.GaussianBlur(radius=alpha))


def jpeg(image, alpha):
    # alpha — качество JPEG (100 = без потерь по сетке).
    if alpha >= 100:
        return image
    buffer = io.BytesIO()
    image.convert("RGB").save(buffer, format="JPEG", quality=int(alpha))
    buffer.seek(0)
    return Image.open(buffer).convert("RGB")


def occlusion(image, alpha):
    # alpha — доля площади кадра под серым квадратом (имитация геля/линейки/волоса).
    if alpha == 0:
        return image
    width, height = image.size
    side = int(((alpha / 100) * width * height) ** 0.5)
    rng = random.Random(42)
    x = rng.randint(0, max(0, width - side))
    y = rng.randint(0, max(0, height - side))
    arr = np.asarray(image).copy()
    arr[y:y + side, x:x + side] = 128
    return Image.fromarray(arr)


# Имя -> (функция, сетка параметров, единица измерения, значение-тождество).
PERTURBATIONS = {
    "brightness": (brightness, [-0.5, -0.25, 0, 0.25, 0.5], "EV", 0),
    "contrast": (contrast, [0.5, 0.75, 1.0, 1.25, 1.5], "factor", 1.0),
    "gamma": (gamma, [0.5, 0.75, 1.0, 1.5, 2.0], "exponent", 1.0),
    "hue": (hue_shift, [0, 8, 16, 32, 64], "degrees", 0),
    "noise": (gaussian_noise, [0, 5, 10, 20, 40], "sigma_uint8", 0),
    "blur": (gaussian_blur, [0, 1, 2, 4, 8], "radius_px", 0),
    "jpeg": (jpeg, [10, 20, 40, 60, 80, 100], "quality", 100),
    "occlusion": (occlusion, [0, 5, 10, 20, 30], "percent_area", 0),
}

# Коридоры, в которых система обязана оставаться практически неизменной.
# За их пределами деградация допустима, но должна быть плавной.
INVARIANT_RANGES = {
    "brightness": [-0.25, 0.25],
    "contrast": [0.75, 1.25],
    "gamma": [0.75, 1.5],
    "hue": [0, 16],
    "jpeg": [80, 100],
}


In [ ]:
%%writefile /kaggle/working/isic_project/src/train.py
"""Обучение бейслайна: ResNet18 + CrossEntropy, без аугментаций и регуляризации.

Бейслайн сознательно «голый», чтобы честно увидеть переобучение и эффект
дисбаланса. После обучения пишутся веса лучшего по val accuracy чекпойнта,
кривые обучения (history.json) и result.json со сводкой прогона.

Запуск:
    python -m src.train --config configs/baseline.yaml
    python -m src.train --name myrun --epochs 8
"""
from pathlib import Path

import click
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from src.config import load_config, num_classes
from src.dataset import ISICDataset, get_splits, pick_device, train_transform
from src.model import build_model
from src.utils import dump_json, ensure_dir, first_set, run_stamp


def _run_epoch(model, loader, device, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    loss_sum, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(is_train):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            loss_sum += loss.item() * labels.size(0)
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)
    return loss_sum / max(total, 1), correct / max(total, 1)


def run_training(cfg, name_override=None):
    device = pick_device()
    print("device:", device)

    train_cfg, model_cfg = cfg["train"], cfg["model"]
    transform = train_transform(model_cfg["image_size"])

    train_df, val_df = get_splits(cfg)
    print("train:", len(train_df), "val:", len(val_df))

    pin = device.type == "cuda"
    train_loader = DataLoader(
        ISICDataset(train_df, cfg["paths"]["images_dir"], transform),
        batch_size=train_cfg["batch_size"], shuffle=True,
        num_workers=train_cfg["num_workers"], pin_memory=pin)
    val_loader = DataLoader(
        ISICDataset(val_df, cfg["paths"]["images_dir"], transform),
        batch_size=train_cfg["batch_size"], shuffle=False,
        num_workers=train_cfg["num_workers"], pin_memory=pin)

    torch.manual_seed(cfg["seed"])
    model = build_model(model_cfg["backbone"], num_classes(cfg), model_cfg["pretrained"]).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=train_cfg["lr"],
                                 weight_decay=train_cfg["weight_decay"])

    runs_dir = ensure_dir(cfg["paths"]["runs_dir"])
    checkpoint_path = Path(cfg["paths"]["checkpoint"])

    history = []
    best_acc, best_epoch = -1.0, 0
    for epoch in range(1, train_cfg["epochs"] + 1):
        tr_loss, tr_acc = _run_epoch(model, train_loader, device, criterion, optimizer)
        val_loss, val_acc = _run_epoch(model, val_loader, device, criterion)
        history.append({"epoch": epoch, "train_loss": tr_loss, "train_acc": tr_acc,
                        "val_loss": val_loss, "val_acc": val_acc})
        print("epoch %d  train_loss=%.4f acc=%.4f | val_loss=%.4f acc=%.4f"
              % (epoch, tr_loss, tr_acc, val_loss, val_acc))
        if val_acc > best_acc:
            best_acc, best_epoch = val_acc, epoch
            torch.save(model.state_dict(), checkpoint_path)

    dump_json(runs_dir / "history.json", history)

    run_name = run_stamp(first_set(name_override, cfg.get("name")))
    result = {
        "run_name": run_name,
        "config_path": cfg["config_path"],
        "backbone": model_cfg["backbone"],
        "epochs": train_cfg["epochs"],
        "best_epoch": best_epoch,
        "best_val_acc": best_acc,
        "checkpoint": str(checkpoint_path),
        "n_train": len(train_df),
        "n_val": len(val_df),
        "history": history,
    }
    result_path = dump_json(runs_dir / "result.json", result)
    print("saved checkpoint:", checkpoint_path, "(epoch", best_epoch, "val_acc=%.4f)" % best_acc)
    return result_path


@click.command(context_settings={"help_option_names": ["-h", "--help"]})
@click.option("--config", type=click.Path(path_type=Path), default=None, help="путь к yaml-конфигу")
@click.option("--name", default=None, help="суффикс имени прогона")
@click.option("--epochs", type=int, default=None)
@click.option("--lr", type=float, default=None)
@click.option("--batch-size", type=int, default=None)
def main(config, name, epochs, lr, batch_size):
    cfg = load_config(config)
    if epochs is not None:
        cfg["train"]["epochs"] = epochs
    if lr is not None:
        cfg["train"]["lr"] = lr
    if batch_size is not None:
        cfg["train"]["batch_size"] = batch_size
    click.echo(run_training(cfg, name))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/isic_project/src/evaluate.py
"""Оценка чекпойнта на валидации: per-class, confusion matrix, ROC, сводка.

Помимо отчётных таблиц модуль сохраняет «сырые» предсказания
(`reports/val_probs.npz` + `reports/val_index.csv`), чтобы калибровка, отбор
картинок-ошибок и графики могли считаться офлайн, без повторного прогона модели.

Запуск:
    python -m src.evaluate --config configs/baseline.yaml
"""
import os
from pathlib import Path

import click
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from torch.utils.data import DataLoader

from src.config import load_config, num_classes
from src.dataset import ISICDataset, eval_transform, get_splits, pick_device
from src.metrics import sweep, threshold_grid, uniform_risk
from src.model import build_model, load_checkpoint
from src.utils import dump_json, ensure_dir


def predict_probs(model, loader, device):
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for images, batch_labels in loader:
            outputs = model(images.to(device))
            probs.append(torch.softmax(outputs, dim=1).cpu().numpy())
            labels.append(batch_labels.numpy())
    return np.vstack(probs), np.concatenate(labels)


def operating_point(scores, positive, decision):
    """Выбор порога-минимизатора риска и сводка по нему."""
    rows = sweep(scores, positive, threshold_grid(decision["threshold_step"]),
                 decision["cost_fp"], decision["cost_fn"])
    best = min(rows, key=lambda r: r["risk"])
    default_counts = next((r for r in rows if abs(r["threshold"] - decision["default_threshold"]) < 1e-9), None)
    return best, default_counts, rows


def run_evaluation(cfg):
    device = pick_device()
    print("device:", device)

    classes = cfg["classes"]
    nc = num_classes(cfg)
    reports_dir = ensure_dir(cfg["paths"]["reports_dir"])

    _, val_df = get_splits(cfg)
    loader = DataLoader(
        ISICDataset(val_df, cfg["paths"]["images_dir"], eval_transform(cfg["model"]["image_size"])),
        batch_size=cfg["eval"]["batch_size"], shuffle=False,
        num_workers=cfg["eval"]["num_workers"], pin_memory=(device.type == "cuda"))

    model = build_model(cfg["model"]["backbone"], nc, pretrained=False).to(device)
    load_checkpoint(model, cfg["paths"]["checkpoint"], device)

    probs, labels = predict_probs(model, loader, device)
    preds = probs.argmax(axis=1)

    # Сырые предсказания для офлайн-аналитики.
    np.savez(os.path.join(reports_dir, "val_probs.npz"), probs=probs, labels=labels)
    val_df.assign(pred=preds, max_prob=probs.max(axis=1))[
        ["image", "label", "pred", "max_prob"]
    ].to_csv(os.path.join(reports_dir, "val_index.csv"), index=False)

    # 1) per-class precision/recall/F1.
    report = classification_report(labels, preds, labels=list(range(nc)),
                                   target_names=classes, output_dict=True, zero_division=0)
    pd.DataFrame(report).T.to_csv(os.path.join(reports_dir, "per_class_metrics.csv"))

    # 2) confusion matrix (абсолютная и построчно-нормированная).
    cm = confusion_matrix(labels, preds, labels=list(range(nc)))
    pd.DataFrame(cm, index=classes, columns=classes).to_csv(os.path.join(reports_dir, "confusion_matrix.csv"))
    row_sums = cm.sum(axis=1, keepdims=True)
    cm_norm = np.divide(cm, row_sums, where=row_sums > 0, out=np.zeros_like(cm, dtype=float))
    pd.DataFrame(cm_norm, index=classes, columns=classes).to_csv(
        os.path.join(reports_dir, "confusion_matrix_normalized.csv"))

    # 3) бинаризация «позитивный класс vs всё» + ROC.
    decision = cfg["decision"]
    pos_idx = decision["positive_index"]
    scores = probs[:, pos_idx]
    positive = (labels == pos_idx).astype(int)

    roc_auc = float(roc_auc_score(positive, scores)) if positive.sum() else float("nan")
    fpr, tpr, thr = roc_curve(positive, scores)
    pd.DataFrame({"fpr": fpr, "tpr": tpr, "threshold": thr}).to_csv(
        os.path.join(reports_dir, "roc_%s.csv" % decision["positive_class"].lower()), index=False)

    best, default_counts, _ = operating_point(scores, positive, decision)
    default_risk = uniform_risk(default_counts, decision["cost_fp"], decision["cost_fn"]) if default_counts else None

    summary = {
        "positive_class": decision["positive_class"],
        "overall_accuracy": float((preds == labels).mean()),
        "macro_f1": float(report["macro avg"]["f1-score"]),
        "weighted_f1": float(report["weighted avg"]["f1-score"]),
        "roc_auc": roc_auc,
        "operating_point": {**best, "cost_fp": decision["cost_fp"], "cost_fn": decision["cost_fn"]},
        "default_threshold": decision["default_threshold"],
        "default_risk": default_risk,
        "risk_gain_vs_default": (round(best["risk"] / default_risk - 1, 4) if default_risk else None),
        "n_val": int(len(labels)),
        "n_positive_val": int(positive.sum()),
    }
    dump_json(os.path.join(reports_dir, "summary.json"), summary)

    print("accuracy=%.3f macro_f1=%.3f roc_auc=%.3f op_tau=%.2f risk=%d"
          % (summary["overall_accuracy"], summary["macro_f1"], roc_auc,
             best["threshold"], best["risk"]))
    return Path(reports_dir) / "summary.json"


@click.command(context_settings={"help_option_names": ["-h", "--help"]})
@click.option("--config", type=click.Path(path_type=Path), default=None, help="путь к yaml-конфигу")
def main(config):
    click.echo(run_evaluation(load_config(config)))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/isic_project/src/data_audit.py
"""Аудит данных и протокола: распределение классов, группировка, утечки.

Фиксирует таблицу `ground_truth.csv` и протокол сплита как инженерный артефакт:
распределение по классам, размер групп `lesion_id`, отсутствие пересечения
групп между train и val, доля малигнантных классов. Числа уходят в
`reports/data_audit.json`; содержательный разбор и реестр рисков — в curated
отчёте `reports/data_audit.md`.

Запуск:
    python -m src.data_audit --config configs/baseline.yaml
"""
import os
from collections import Counter
from pathlib import Path

import click

from src.config import load_config
from src.dataset import load_table, split_frame
from src.utils import dump_json


def group_overlap(train_df, val_df, group_by):
    if group_by not in train_df.columns:
        return {"available": False}
    train_groups = set(train_df[group_by].astype(str))
    val_groups = set(val_df[group_by].astype(str))
    overlap = train_groups & val_groups
    return {
        "available": True,
        "train_groups": len(train_groups),
        "val_groups": len(val_groups),
        "overlapping_groups": len(overlap),
        "overlap_examples": sorted(overlap)[:10],
    }


def collect_stats(cfg):
    classes = cfg["classes"]
    decision = cfg["decision"]
    frame = load_table(cfg["paths"]["ground_truth"])

    label_counts = Counter(int(x) for x in frame["label"])
    class_counts = {name: label_counts.get(i, 0) for i, name in enumerate(classes)}
    total = len(frame)

    group_by = cfg["split"]["group_by"]
    if group_by in frame.columns:
        group_sizes = Counter(frame[group_by].astype(str))
        n_groups = len(group_sizes)
        multi = sum(1 for v in group_sizes.values() if v > 1)
        max_group = max(group_sizes.values()) if group_sizes else 0
    else:
        n_groups, multi, max_group = total, 0, 1

    train_df, val_df = split_frame(frame, group_by, cfg["split"]["val_size"], cfg["split"]["random_state"])

    malignant = decision.get("malignant_classes", [])
    malignant_boxes = sum(class_counts[c] for c in malignant)

    sorted_classes = sorted(class_counts.items(), key=lambda kv: kv[1], reverse=True)
    return {
        "ground_truth": cfg["paths"]["ground_truth"],
        "n_rows": total,
        "n_classes": len(classes),
        "class_counts": class_counts,
        "class_fractions": {k: round(v / max(1, total), 4) for k, v in class_counts.items()},
        "dominant_class": sorted_classes[0][0] if sorted_classes else None,
        "rarest_class": sorted_classes[-1][0] if sorted_classes else None,
        "imbalance_ratio": round(sorted_classes[0][1] / max(1, sorted_classes[-1][1]), 2) if sorted_classes else None,
        "malignant_classes": malignant,
        "malignant_fraction": round(malignant_boxes / max(1, total), 4),
        "grouping": {
            "group_by": group_by,
            "n_groups": n_groups,
            "multi_frame_groups": multi,
            "max_group_size": max_group,
            "avg_frames_per_group": round(total / max(1, n_groups), 3),
        },
        "split": {
            "val_size": cfg["split"]["val_size"],
            "random_state": cfg["split"]["random_state"],
            "n_train": len(train_df),
            "n_val": len(val_df),
            "train_class_counts": {name: int((train_df["label"] == i).sum()) for i, name in enumerate(classes)},
            "val_class_counts": {name: int((val_df["label"] == i).sum()) for i, name in enumerate(classes)},
        },
        "group_overlap": group_overlap(train_df, val_df, group_by),
    }


@click.command(context_settings={"help_option_names": ["-h", "--help"]})
@click.option("--config", type=click.Path(path_type=Path), default=None, help="путь к yaml-конфигу")
def main(config):
    cfg = load_config(config)
    stats = collect_stats(cfg)
    out = dump_json(os.path.join(cfg["paths"]["reports_dir"], "data_audit.json"), stats)
    print("rows=%d classes=%d dominant=%s imbalance=%.1fx groups=%d overlap=%s"
          % (stats["n_rows"], stats["n_classes"], stats["dominant_class"],
             stats["imbalance_ratio"] or 0, stats["grouping"]["n_groups"],
             stats["group_overlap"].get("overlapping_groups")))
    click.echo(out)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/isic_project/src/thresholds.py
"""Перебор порога и выбор рабочей точки для критичного класса.

Читает сохранённые предсказания (`reports/val_probs.npz`), бинаризует задачу
«позитивный класс vs всё» и прогоняет сетку порогов, минимизируя асимметричную
функцию риска `Risk(tau) = c_fp*FP + c_fn*FN`. Дополнительно строит сенсити-сweep
для объединённого малигнантного класса (MEL∪BCC∪SCC). Машинные таблицы —
`threshold_sweep.csv/json` и `operating_point.json`; интерпретация — в
`reports/operating_point.md`.

Запуск:
    python -m src.thresholds --config configs/baseline.yaml
"""
import csv
import os
from pathlib import Path

import click
import numpy as np

from src.config import load_config
from src.metrics import sweep, threshold_grid, uniform_risk
from src.utils import dump_json, load_json


def load_probs(reports_dir):
    data = np.load(os.path.join(reports_dir, "val_probs.npz"))
    return data["probs"], data["labels"]


def write_sweep_csv(rows, path):
    cols = ["threshold", "tp", "fp", "fn", "tn", "recall", "precision", "specificity", "fpr", "f1", "risk"]
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=cols)
        writer.writeheader()
        for row in rows:
            writer.writerow({k: row[k] for k in cols})


def run_thresholds(cfg):
    reports_dir = cfg["paths"]["reports_dir"]
    probs, labels = load_probs(reports_dir)

    decision = cfg["decision"]
    taus = threshold_grid(decision["threshold_step"])
    c_fp, c_fn = decision["cost_fp"], decision["cost_fn"]

    # Основная задача: позитивный класс (меланома) vs всё.
    pos_idx = decision["positive_index"]
    pos_scores = probs[:, pos_idx]
    positive = (labels == pos_idx).astype(int)
    rows = sweep(pos_scores, positive, taus, c_fp, c_fn)
    write_sweep_csv(rows, os.path.join(reports_dir, "threshold_sweep.csv"))

    best = min(rows, key=lambda r: r["risk"])
    default = next((r for r in rows if abs(r["threshold"] - decision["default_threshold"]) < 1e-9), None)
    default_risk = uniform_risk(default, c_fp, c_fn) if default else None

    # Сенсити: объединённый малигнантный класс vs доброкачественный.
    mal_idx = decision["malignant_indices"]
    mal_scores = probs[:, mal_idx].sum(axis=1)
    mal_positive = np.isin(labels, mal_idx).astype(int)
    mal_rows = sweep(mal_scores, mal_positive, taus, c_fp, c_fn)
    mal_best = min(mal_rows, key=lambda r: r["risk"])

    payload = {
        "positive_class": decision["positive_class"],
        "cost_fp": c_fp,
        "cost_fn": c_fn,
        "operating_point": best,
        "default_threshold": decision["default_threshold"],
        "default_point": default,
        "default_risk": default_risk,
        "risk_gain_vs_default": (round(best["risk"] / default_risk - 1, 4) if default_risk else None),
        "rows": rows,
        "malignant_sensitivity": {
            "classes": decision["malignant_classes"],
            "operating_point": mal_best,
            "rows": mal_rows,
        },
    }
    dump_json(os.path.join(reports_dir, "threshold_sweep.json"), payload)
    dump_json(os.path.join(reports_dir, "operating_point.json"), {
        "positive_class": decision["positive_class"],
        "cost_fp": c_fp, "cost_fn": c_fn,
        "operating_point": best,
        "default_point": default,
        "risk_gain_vs_default": payload["risk_gain_vs_default"],
    })

    print("operating tau=%.2f recall=%.3f precision=%.3f risk=%d (gain vs default: %s)"
          % (best["threshold"], best["recall"], best["precision"], best["risk"],
             payload["risk_gain_vs_default"]))
    return Path(reports_dir) / "operating_point.json"


@click.command(context_settings={"help_option_names": ["-h", "--help"]})
@click.option("--config", type=click.Path(path_type=Path), default=None, help="путь к yaml-конфигу")
def main(config):
    click.echo(run_thresholds(load_config(config)))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/isic_project/src/calibration.py
"""Калибровка вероятностей: ECE и reliability-бины.

Отвечает на вопрос «можно ли читать выходной скор как вероятность правоты».
Считает два разреза: (1) многоклассовый — max-softmax против факта верности
arg-max; (2) бинарный — скор позитивного класса против факта его наличия.
Результат — `reports/calibration.json`, который затем рисует `figures.py`.

Запуск:
    python -m src.calibration --config configs/baseline.yaml
"""
import os
from pathlib import Path

import click
import numpy as np

from src.config import load_config
from src.metrics import reliability_bins
from src.utils import dump_json


def run_calibration(cfg):
    reports_dir = cfg["paths"]["reports_dir"]
    data = np.load(os.path.join(reports_dir, "val_probs.npz"))
    probs, labels = data["probs"], data["labels"]
    n_bins = cfg["calibration"]["n_bins"]

    # (1) многоклассовая уверенность arg-max.
    preds = probs.argmax(axis=1)
    max_conf = probs.max(axis=1)
    correct = (preds == labels).astype(int)
    multiclass = reliability_bins(max_conf, correct, n_bins)

    # (2) бинарный скор позитивного класса.
    pos_idx = cfg["decision"]["positive_index"]
    pos_scores = probs[:, pos_idx]
    pos_true = (labels == pos_idx).astype(int)
    binary = reliability_bins(pos_scores, pos_true, n_bins)

    payload = {
        "n_bins": n_bins,
        "positive_class": cfg["decision"]["positive_class"],
        "multiclass": multiclass,
        "binary_positive": binary,
        # diagram-совместимый верхний уровень: используем многоклассовый разрез.
        "ece": multiclass["ece"],
        "bins": multiclass["bins"],
    }
    out = dump_json(os.path.join(reports_dir, "calibration.json"), payload)
    print("ECE multiclass=%.4f  ECE %s=%.4f"
          % (multiclass["ece"], cfg["decision"]["positive_class"], binary["ece"]))
    click.echo(out)
    return out


@click.command(context_settings={"help_option_names": ["-h", "--help"]})
@click.option("--config", type=click.Path(path_type=Path), default=None, help="путь к yaml-конфигу")
def main(config):
    run_calibration(load_config(config))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/isic_project/src/subgroups.py
"""Worst-slice анализ: где модель проседает по подгруппам данных.

В отличие от стресс-теста (искажения входа), здесь режутся естественные
подгруппы валидации — по классу, по «злокачественности», по уверенности модели
и по размеру группы кадров одного поражения. Для каждого среза считаются
support / accuracy / macro-F1 / recall критичного класса. Таблица сортируется по
качеству (худшие сверху) и пишется в `reports/slice_metrics.csv`.

Запуск:
    python -m src.subgroups --config configs/baseline.yaml
"""
import os
from pathlib import Path

import click
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

from src.config import load_config
from src.dataset import load_table


def _slice_metrics(sub, pos_idx, n_classes):
    y_true = sub["label"].to_numpy()
    y_pred = sub["pred"].to_numpy()
    acc = float((y_true == y_pred).mean()) if len(sub) else 0.0
    macro_f1 = float(f1_score(y_true, y_pred, labels=list(range(n_classes)),
                              average="macro", zero_division=0)) if len(sub) else 0.0
    pos_mask = y_true == pos_idx
    pos_support = int(pos_mask.sum())
    pos_recall = float((y_pred[pos_mask] == pos_idx).mean()) if pos_support else None
    return {"support": len(sub), "accuracy": round(acc, 4), "macro_f1": round(macro_f1, 4),
            "pos_support": pos_support,
            "pos_recall": round(pos_recall, 4) if pos_recall is not None else None}


def build_slices(index, classes, decision):
    pos_idx = decision["positive_index"]
    mal_idx = set(decision["malignant_indices"])
    nc = len(classes)
    rows = []

    # по истинному классу
    for i, name in enumerate(classes):
        sub = index[index["label"] == i]
        if len(sub):
            rows.append({"slice": f"class:{name}", **_slice_metrics(sub, pos_idx, nc)})

    # злокачественные vs доброкачественные
    for label, mask in (("group:malignant", index["label"].isin(mal_idx)),
                        ("group:benign", ~index["label"].isin(mal_idx))):
        sub = index[mask]
        if len(sub):
            rows.append({"slice": label, **_slice_metrics(sub, pos_idx, nc)})

    # по уверенности arg-max
    bands = [("conf:low(<0.5)", index["max_prob"] < 0.5),
             ("conf:mid(0.5-0.8)", (index["max_prob"] >= 0.5) & (index["max_prob"] < 0.8)),
             ("conf:high(>=0.8)", index["max_prob"] >= 0.8)]
    for label, mask in bands:
        sub = index[mask]
        if len(sub):
            rows.append({"slice": label, **_slice_metrics(sub, pos_idx, nc)})

    # по размеру группы кадров одного поражения
    if "group_size" in index.columns:
        for label, mask in (("lesion:singleton", index["group_size"] == 1),
                            ("lesion:multi_frame", index["group_size"] > 1)):
            sub = index[mask]
            if len(sub):
                rows.append({"slice": label, **_slice_metrics(sub, pos_idx, nc)})

    return rows


def run_subgroups(cfg):
    reports_dir = cfg["paths"]["reports_dir"]
    index = pd.read_csv(os.path.join(reports_dir, "val_index.csv"))

    # подтянем размер группы поражения из ground_truth
    gt = load_table(cfg["paths"]["ground_truth"])
    group_by = cfg["split"]["group_by"]
    if group_by in gt.columns:
        sizes = gt[group_by].astype(str).value_counts()
        name_to_group = dict(zip(gt["image"].astype(str), gt[group_by].astype(str)))
        index["group_size"] = index["image"].astype(str).map(name_to_group).map(sizes).fillna(1).astype(int)

    rows = build_slices(index, cfg["classes"], cfg["decision"])
    rows.sort(key=lambda r: r["accuracy"])

    out = os.path.join(reports_dir, "slice_metrics.csv")
    pd.DataFrame(rows, columns=["slice", "support", "accuracy", "macro_f1", "pos_support", "pos_recall"]).to_csv(out, index=False)
    print("worst slices:", [(r["slice"], r["accuracy"]) for r in rows[:3]])
    click.echo(out)
    return out


@click.command(context_settings={"help_option_names": ["-h", "--help"]})
@click.option("--config", type=click.Path(path_type=Path), default=None, help="путь к yaml-конфигу")
def main(config):
    run_subgroups(load_config(config))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/isic_project/src/error_cases.py
"""Отбор и визуализация характерных ошибок по критичному классу.

Для бинарного решения «меланома vs всё» на рабочем пороге каждый кадр валидации
относится к TP / FP / FN / TN. Из каждой категории сохраняется набор примеров
(половина — самые «вопиющие» по уверенности, половина — случайные) с подписью
истинного и предсказанного класса и скором. Картинки идут в
`reports/error_cases/<категория>/`, состав выборки — в `selection.json`.

Запуск:
    python -m src.error_cases --config configs/baseline.yaml --per-category 12
"""
import os
import random
from pathlib import Path

import click
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont

from src.config import load_config
from src.utils import dump_json, ensure_dir, load_json

GREEN, RED = "#20a050", "#d04040"


def _font():
    try:
        return ImageFont.load_default()
    except Exception:
        return None


def _resolve_image(images_dir, name):
    for suffix in (".jpg", ".jpeg", ".png"):
        candidate = os.path.join(images_dir, name + suffix)
        if os.path.isfile(candidate):
            return candidate
    return os.path.join(images_dir, name)


def _annotate(image_path, lines, color):
    image = Image.open(image_path).convert("RGB")
    if max(image.size) < 256:                       # demo-картинки крошечные — увеличим
        scale = 256 // max(1, min(image.size)) + 1
        image = image.resize((image.width * scale, image.height * scale))
    draw = ImageDraw.Draw(image)
    font = _font()
    y = 2
    for text in lines:
        box = draw.textbbox((4, y), text, font=font)
        draw.rectangle((box[0] - 2, box[1] - 1, box[2] + 2, box[3] + 1), fill=color)
        draw.text((4, y), text, fill="white", font=font)
        y = box[3] + 3
    return image


def _operating_tau(cfg):
    op_path = os.path.join(cfg["paths"]["reports_dir"], "operating_point.json")
    if os.path.isfile(op_path):
        return float(load_json(op_path)["operating_point"]["threshold"])
    return float(cfg["decision"]["default_threshold"])


def run_error_cases(cfg, per_category=12):
    classes = cfg["classes"]
    reports_dir = cfg["paths"]["reports_dir"]
    images_dir = cfg["paths"]["images_dir"]
    pos_idx = cfg["decision"]["positive_index"]
    pos_name = cfg["decision"]["positive_class"]

    index = pd.read_csv(os.path.join(reports_dir, "val_index.csv"))
    probs = np.load(os.path.join(reports_dir, "val_probs.npz"))["probs"]
    pos_scores = probs[:, pos_idx]
    tau = _operating_tau(cfg)

    buckets = {"tp": [], "fp": [], "fn": [], "tn": []}
    for i, row in index.iterrows():
        is_pos = int(row["label"]) == pos_idx
        emitted = pos_scores[i] >= tau
        kind = ("tp" if emitted and is_pos else "fp" if emitted and not is_pos
                else "fn" if (not emitted) and is_pos else "tn")
        buckets[kind].append({
            "image": str(row["image"]),
            "true": classes[int(row["label"])],
            "pred": classes[int(row["pred"])],
            "score": float(pos_scores[i]),
        })

    rng = random.Random(cfg["seed"])
    out_root = ensure_dir(os.path.join(reports_dir, "error_cases"))
    selection = {}
    for kind, items in buckets.items():
        cat_dir = ensure_dir(os.path.join(out_root, kind))
        for old in Path(cat_dir).glob("*.jpg"):
            old.unlink()
        # самые уверенные сверху + случайные для разнообразия
        ordered = sorted(items, key=lambda c: c["score"], reverse=(kind != "fn"))
        k_top = per_category // 2
        chosen = ordered[:k_top]
        rest = [c for c in items if c not in chosen]
        if rest:
            chosen += rng.sample(rest, min(per_category - k_top, len(rest)))
        color = GREEN if kind in ("tp", "tn") else RED
        for c in chosen:
            lines = [f"{kind.upper()}  {pos_name}={c['score']:.2f}", f"true={c['true']} pred={c['pred']}"]
            img = _annotate(_resolve_image(images_dir, c["image"]), lines, color)
            img.save(os.path.join(cat_dir, c["image"] + ".jpg"))
        selection[kind] = {"available": len(items), "selected": [c["image"] for c in chosen]}

    dump_json(os.path.join(out_root, "selection.json"),
              {"tau": tau, "positive_class": pos_name, "per_category": per_category, "categories": selection})
    print("error cases @tau=%.2f:" % tau,
          {k: selection[k]["available"] for k in buckets})
    return Path(out_root)


@click.command(context_settings={"help_option_names": ["-h", "--help"]})
@click.option("--config", type=click.Path(path_type=Path), default=None, help="путь к yaml-конфигу")
@click.option("--per-category", type=int, default=12)
def main(config, per_category):
    click.echo(run_error_cases(load_config(config), per_category))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/isic_project/src/figures.py
"""Отрисовка отчётных графиков из готовых таблиц reports/.

Каждая функция самостоятельна и мягко пропускается, если её исходные данные не
сгенерированы. Графики складываются в `reports/figures/`:
    training_curves.png      — кривые обучения (history.json)
    roc_<class>.png          — ROC критичного класса (roc_*.csv)
    precision_recall.png     — PR из threshold_sweep.csv
    metrics_vs_threshold.png — precision/recall/F1/specificity по tau
    risk_vs_threshold.png    — функция риска по tau с argmin
    confusion_matrix.png     — нормированная матрица ошибок
    stress_degradation.png   — деградация macro-F1 по стресс-сценариям
    reliability_diagram.png  — калибровка (calibration.json)

Запуск:
    python -m src.figures --config configs/baseline.yaml
"""
import json
import os
from pathlib import Path

import click
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config import load_config
from src.utils import ensure_dir


def _save(fig, path):
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)
    print("wrote", path)


def training_curves(reports_dir, runs_dir, out):
    history_path = os.path.join(runs_dir, "history.json")
    if not os.path.isfile(history_path):
        return
    hist = json.loads(Path(history_path).read_text())
    epochs = [h["epoch"] for h in hist]
    fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(11, 4.2))
    ax_loss.plot(epochs, [h["train_loss"] for h in hist], marker="o", label="train")
    ax_loss.plot(epochs, [h["val_loss"] for h in hist], marker="o", label="val")
    ax_loss.set_title("Loss по эпохам"); ax_loss.set_xlabel("эпоха"); ax_loss.set_ylabel("loss")
    ax_loss.grid(alpha=0.3); ax_loss.legend()
    ax_acc.plot(epochs, [h["train_acc"] for h in hist], marker="o", label="train")
    ax_acc.plot(epochs, [h["val_acc"] for h in hist], marker="o", label="val")
    ax_acc.set_title("Accuracy по эпохам"); ax_acc.set_xlabel("эпоха"); ax_acc.set_ylabel("accuracy")
    ax_acc.grid(alpha=0.3); ax_acc.legend()
    _save(fig, out)


def roc_curve_fig(reports_dir, positive_class, out):
    path = os.path.join(reports_dir, "roc_%s.csv" % positive_class.lower())
    if not os.path.isfile(path):
        return
    df = pd.read_csv(path)
    fig, ax = plt.subplots(figsize=(5.5, 5))
    ax.plot(df["fpr"], df["tpr"], linewidth=1.6)
    ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=0.8)
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR (recall)")
    ax.set_title(f"ROC: {positive_class} vs всё")
    ax.grid(alpha=0.3); ax.set_xlim(0, 1); ax.set_ylim(0, 1.05)
    _save(fig, out)


def _load_sweep(reports_dir):
    path = os.path.join(reports_dir, "threshold_sweep.csv")
    return pd.read_csv(path) if os.path.isfile(path) else None


def precision_recall_fig(reports_dir, out):
    df = _load_sweep(reports_dir)
    if df is None:
        return
    fig, ax = plt.subplots(figsize=(5.5, 5))
    ax.plot(df["recall"], df["precision"], marker=".", linewidth=1.3)
    ax.set_xlabel("recall"); ax.set_ylabel("precision")
    ax.set_title("Precision vs recall (по порогу)")
    ax.grid(alpha=0.3); ax.set_xlim(0, 1); ax.set_ylim(0, 1.05)
    _save(fig, out)


def metrics_vs_threshold_fig(reports_dir, out):
    df = _load_sweep(reports_dir)
    if df is None:
        return
    fig, ax = plt.subplots(figsize=(7, 4.6))
    for col in ("precision", "recall", "f1", "specificity"):
        if col in df.columns:
            ax.plot(df["threshold"], df[col], linewidth=1.4, label=col)
    ax.set_xlabel("порог tau"); ax.set_ylabel("значение метрики")
    ax.set_title("Метрики по порогу"); ax.grid(alpha=0.3); ax.legend()
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.05)
    _save(fig, out)


def risk_vs_threshold_fig(reports_dir, out):
    df = _load_sweep(reports_dir)
    if df is None:
        return
    tau = df["threshold"].tolist()
    risk = df["risk"].tolist()
    argmin = tau[int(np.argmin(risk))]
    fig, ax = plt.subplots(figsize=(7, 4.6))
    ax.plot(tau, risk, marker=".", color="C3", linewidth=1.4)
    ax.axvline(argmin, color="C0", linestyle="--", linewidth=0.9)
    ax.text(argmin, max(risk) * 0.95, f"  argmin tau={argmin}", color="C0")
    ax.set_xlabel("порог tau"); ax.set_ylabel("risk = c_fp·FP + c_fn·FN")
    ax.set_title("Функция риска по порогу"); ax.grid(alpha=0.3); ax.set_xlim(0, 1)
    _save(fig, out)


def confusion_fig(reports_dir, out):
    path = os.path.join(reports_dir, "confusion_matrix_normalized.csv")
    if not os.path.isfile(path):
        return
    df = pd.read_csv(path, index_col=0)
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    im = ax.imshow(df.to_numpy(), cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(df.columns))); ax.set_xticklabels(df.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(df.index))); ax.set_yticklabels(df.index)
    ax.set_xlabel("предсказание"); ax.set_ylabel("истина")
    ax.set_title("Confusion matrix (нормированная по строкам)")
    for i in range(df.shape[0]):
        for j in range(df.shape[1]):
            v = df.to_numpy()[i, j]
            if v >= 0.01:
                ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                        color="white" if v > 0.5 else "black", fontsize=7)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    _save(fig, out)


def stress_fig(reports_dir, out):
    # совместимость: старый stress хранится в stress_metrics.csv (scenario,...) либо
    # новый — те же колонки perturbation/alpha. Рисуем то, что нашли.
    path = os.path.join(reports_dir, "stress_metrics.csv")
    if not os.path.isfile(path):
        return
    df = pd.read_csv(path)
    fig, ax = plt.subplots(figsize=(8, 4.6))
    if "scenario" in df.columns:
        x = df["scenario"]
        ax.bar(x, df["macro_f1"], alpha=0.85)
        ax.set_ylabel("macro-F1"); ax.set_title("Стресс-сценарии: macro-F1")
        ax.set_xticklabels(x, rotation=30, ha="right")
    else:
        for name, grp in df.groupby("perturbation"):
            ax.plot(grp["alpha"], grp["macro_f1"], marker="o", label=name)
        ax.set_xlabel("alpha"); ax.set_ylabel("macro-F1")
        ax.set_title("Деградация macro-F1 по искажениям"); ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    _save(fig, out)


def reliability_fig(reports_dir, out):
    path = os.path.join(reports_dir, "calibration.json")
    if not os.path.isfile(path):
        return
    data = json.loads(Path(path).read_text())
    bins = [b for b in data["bins"] if b["count"] > 0]
    if not bins:
        return
    centers = [(b["low"] + b["high"]) / 2 for b in bins]
    accs = [b["acc"] for b in bins]
    confs = [b["conf"] for b in bins]
    width = (bins[0]["high"] - bins[0]["low"]) * 0.9
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.bar(centers, accs, width=width, alpha=0.7, edgecolor="black", label="accuracy")
    ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=0.8, label="идеальная калибровка")
    ax.scatter(centers, confs, color="red", marker="x", s=50, zorder=3, label="средняя уверенность")
    ax.set_xlabel("предсказанная уверенность"); ax.set_ylabel("accuracy")
    ax.set_title(f"Reliability diagram (ECE = {data['ece']:.3f})")
    ax.grid(alpha=0.3); ax.set_xlim(0, 1); ax.set_ylim(0, 1.05); ax.legend(loc="upper left")
    _save(fig, out)


def run_figures(cfg):
    reports_dir = cfg["paths"]["reports_dir"]
    runs_dir = cfg["paths"]["runs_dir"]
    positive_class = cfg["decision"]["positive_class"]
    figures_dir = ensure_dir(os.path.join(reports_dir, "figures"))

    training_curves(reports_dir, runs_dir, os.path.join(figures_dir, "training_curves.png"))
    roc_curve_fig(reports_dir, positive_class, os.path.join(figures_dir, "roc_%s.png" % positive_class.lower()))
    precision_recall_fig(reports_dir, os.path.join(figures_dir, "precision_recall.png"))
    metrics_vs_threshold_fig(reports_dir, os.path.join(figures_dir, "metrics_vs_threshold.png"))
    risk_vs_threshold_fig(reports_dir, os.path.join(figures_dir, "risk_vs_threshold.png"))
    confusion_fig(reports_dir, os.path.join(figures_dir, "confusion_matrix.png"))
    stress_fig(reports_dir, os.path.join(figures_dir, "stress_degradation.png"))
    reliability_fig(reports_dir, os.path.join(figures_dir, "reliability_diagram.png"))
    return Path(figures_dir)


@click.command(context_settings={"help_option_names": ["-h", "--help"]})
@click.option("--config", type=click.Path(path_type=Path), default=None, help="путь к yaml-конфигу")
def main(config):
    click.echo(run_figures(load_config(config)))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/isic_project/scripts/prepare_isic2019.py
# Конвертация официальных файлов ISIC 2019 в формат проекта.
# На вход:
#   ISIC_2019_Training_GroundTruth.csv — image, MEL, NV, BCC, AK, BKL, DF, VASC, SCC, UNK
#   ISIC_2019_Training_Metadata.csv    — image, ..., lesion_id (опционально)
# На выход: ground_truth.csv с колонками image, label, lesion_id
import argparse
import os
import sys

import pandas as pd

CLASS_ORDER = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--gt", required=True, help="ISIC_2019_Training_GroundTruth.csv")
    parser.add_argument("--metadata", default=None, help="ISIC_2019_Training_Metadata.csv (опционально)")
    parser.add_argument("--out", required=True, help="путь к итоговому ground_truth.csv")
    args = parser.parse_args()

    gt = pd.read_csv(args.gt)
    missing = [c for c in CLASS_ORDER if c not in gt.columns]
    if missing:
        print("в GT отсутствуют колонки:", missing, file=sys.stderr)
        sys.exit(1)

    # удаляем UNK (не входит в 8 целевых классов)
    if "UNK" in gt.columns:
        gt = gt[gt["UNK"] < 0.5].copy()

    labels = gt[CLASS_ORDER].to_numpy().argmax(axis=1)
    out = pd.DataFrame({"image": gt["image"].astype(str), "label": labels})

    if args.metadata and os.path.isfile(args.metadata):
        meta = pd.read_csv(args.metadata)
        if "lesion_id" in meta.columns:
            out = out.merge(meta[["image", "lesion_id"]], on="image", how="left")
        else:
            print("в metadata нет lesion_id — сплит будет по image", file=sys.stderr)
    else:
        print("metadata не передана — сплит будет по image", file=sys.stderr)

    if "lesion_id" not in out.columns:
        out["lesion_id"] = out["image"]
    else:
        out["lesion_id"] = out["lesion_id"].fillna(out["image"])

    os.makedirs(os.path.dirname(args.out) or ".", exist_ok=True)
    out.to_csv(args.out, index=False)

    print("saved:", args.out)
    print("n_rows:", len(out))
    print("распределение классов:")
    counts = out["label"].value_counts().sort_index()
    for i, name in enumerate(CLASS_ORDER):
        print("  %d %-5s %6d" % (i, name, int(counts.get(i, 0))))
    n_groups = out["lesion_id"].nunique()
    print("уникальных lesion_id:", n_groups)


if __name__ == "__main__":
    main()


In [ ]:
# автопоиск путей к ISIC 2019 внутри /kaggle/input
import os

print("=== содержимое /kaggle/input ===")
for root, dirs, files in os.walk("/kaggle/input"):
    if root.count("/") - 2 > 3:
        continue
    print(root)
    for f in sorted(files)[:10]:
        print("  ", f)

def find_one(tokens):
    out = []
    for root, _, files in os.walk("/kaggle/input"):
        for name in files:
            low = name.lower()
            if all(t in low for t in tokens):
                out.append(os.path.join(root, name))
    return out

gt_candidates = find_one(["groundtruth", ".csv"]) or find_one(["ground_truth", ".csv"]) or find_one(["labels", ".csv"])
meta_candidates = find_one(["metadata", ".csv"]) or find_one(["meta", ".csv"])
img_candidates = sorted({root for root, _, files in os.walk("/kaggle/input")
                         if len([f for f in files if f.lower().endswith(".jpg")]) > 1000})

print("\nGT:", gt_candidates)
print("META:", meta_candidates)
print("IMAGES:", img_candidates)
assert gt_candidates, "Не найден файл разметки ISIC 2019 GroundTruth CSV"
assert img_candidates, "Не найдена папка с изображениями (>1000 .jpg)"

GT_PATH, IMG_SRC = gt_candidates[0], img_candidates[0]
META_PATH = meta_candidates[0] if meta_candidates else ""

# симлинк на изображения (не копируем ~9 ГБ)
link = "/kaggle/working/isic_project/data/images"
if os.path.islink(link):
    os.unlink(link)
if not os.path.lexists(link):
    os.symlink(IMG_SRC, link)
print("images ->", os.readlink(link) if os.path.islink(link) else link)

os.environ["ISIC_DATA_DIR"] = "/kaggle/working/isic_project/data"
os.environ["ISIC_RUNS_DIR"] = "/kaggle/working/isic_project/runs"
os.environ["ISIC_REPORTS_DIR"] = "/kaggle/working/isic_project/reports"


In [ ]:
meta_arg = ("--metadata " + META_PATH) if META_PATH else ""
!python scripts/prepare_isic2019.py --gt "$GT_PATH" $meta_arg --out data/ground_truth.csv


In [ ]:
# полный пайплайн на едином конфиге
!python -m src.data_audit
!python -m src.train
!python -m src.evaluate
!python -m src.thresholds
!python -m src.calibration
!python -m src.subgroups
!python -m src.error_cases
!python -m src.stress_test
!python -m src.figures


In [ ]:
import json, pandas as pd, os
rep = "/kaggle/working/isic_project/reports"
run = "/kaggle/working/isic_project/runs"

print("=== summary.json ===")
print(json.dumps(json.load(open(os.path.join(rep, "summary.json"))), indent=2, ensure_ascii=False))
print("\n=== per_class_metrics ===")
print(pd.read_csv(os.path.join(rep, "per_class_metrics.csv")).to_string())
print("\n=== slice_metrics (worst-slice) ===")
print(pd.read_csv(os.path.join(rep, "slice_metrics.csv")).to_string())
print("\n=== stress_metrics ===")
print(pd.read_csv(os.path.join(rep, "stress_metrics.csv")).to_string())
print("\n=== operating_point.json ===")
print(json.dumps(json.load(open(os.path.join(rep, "operating_point.json"))), indent=2, ensure_ascii=False))


In [ ]:
import shutil
shutil.make_archive("/kaggle/working/isic_outputs", "zip", "/kaggle/working/isic_project")
print("готово: /kaggle/working/isic_outputs.zip")


## После прогона

1. Скачать `/kaggle/working/isic_outputs.zip`.
2. Распаковать локально — содержимое `reports/` и `runs/` заменит заготовки.
3. Сверить числа в `reports/*.md` с обновлёнными `summary.json`,
   `threshold_sweep.csv`, `calibration.json`.
